In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

# 🚀 The Full Picture

*Part of the Vizuara series — The Good and Evil of the Key-Value Cache*
*Estimated time: 20–30 minutes*

Welcome back. In this final notebook of the series, we step back from the individual mechanics and trace the full arc of what we have built together — from the raw inefficiency of naive transformer inference, through the elegance of the KV cache, all the way to the hardware realities that govern where the bottleneck truly lives.

## 1. Why Does This Matter?

Every time a large language model generates a single token for you — one word, one piece of punctuation — a cascade of decisions is made at the hardware level. Understanding those decisions is the difference between being a *user* of these systems and being an *engineer* who can improve them.

Let us paint the full picture in one motivating question:

> **If the KV cache saves so much computation, why are long-context LLM deployments still expensive and slow?**

By the end of this notebook, you will be able to answer that question precisely, with numbers. We will simulate the complete inference timeline, measure arithmetic intensity, place it on the roofline, and see exactly where the pressure accumulates.

Here is a teaser of what our final output will look like:

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Teaser: the roofline diagram we will build toward ──
fig, ax = plt.subplots(figsize=(9, 5))
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Arithmetic Intensity  (FLOPs / byte)", fontsize=13)
ax.set_ylabel("Achieved Performance  (GFLOPs/s)", fontsize=13)
ax.set_title("Roofline Model — Teaser\n(We will fill this in properly later)", fontsize=13)

x = np.logspace(-1, 3, 400)
bandwidth_GBs = 2_000        # GB/s  (A100-class)
peak_GFLOPS   = 312_000      # GFLOPS (A100 BF16)
roof = np.minimum(bandwidth_GBs * x, peak_GFLOPS)

ax.plot(x, roof, "k-", lw=2.5, label="Hardware roofline")
ax.axvline(x=1.0, color="steelblue", ls="--", alpha=0.5)
ax.scatter([0.3], [bandwidth_GBs * 0.3], s=200, color="crimson",
           zorder=5, label="??? — find out soon")
ax.legend(fontsize=11); ax.grid(True, which="both", alpha=0.3)
plt.tight_layout(); plt.savefig("teaser_roofline.png", dpi=120)
plt.show()
print("Teaser rendered. Let us build up to the real version together.")

## 2. Building Intuition

Before we touch a single line of code, let us reconstruct the entire story in plain English. Think of this as a spoken lecture — close your eyes and picture it.

---

### 🏗️ The Three-Act Story

**Act I — The Villain: Naive Autoregressive Inference**

A language model is given a context of $L$ tokens and must produce one new token. In the naive approach, it runs the full transformer forward pass over all $L$ tokens every single time. The attention mechanism computes relationships between *every pair* of tokens. That is $L^2$ interactions. Generate 100 new tokens from a 1,000-token context? That is 100 separate full passes, each looking at a growing context. The work compounds quadratically. For a 2,000-token contract generating 200 words, we are talking about hundreds of millions of redundant multiplications.

**Act II — The Hero: The KV Cache**

Here is the key insight the KV cache exploits: *the keys and values for all previous tokens do not change*. Once token 47 has been processed, its key vector and value vector are fixed forever — they depend only on token 47's embedding, not on future tokens. So why recompute them? The KV cache simply stores those vectors in memory after the first computation and *reads* them back on every subsequent step instead of recomputing them.

Generation time drops from $O(L^2)$ operations to roughly $O(L)$ per new token. This is a massive win.

**Act III — The Price: The Roofline**

But storing all those cached vectors requires *memory*, and reading them back on every generation step creates a *memory bandwidth* demand. The GPU now spends most of its time waiting for bytes to travel from VRAM to the compute cores — not actually computing. We have traded an arithmetic problem for a memory problem. On the roofline model, this places LLM inference deep in the *memory-bound* regime.

---

### 🚛 The Shipyard Analogy

Imagine a shipyard (the GPU's compute cores) that can assemble 1,000 units per second. Delivery trucks (the memory bus) bring components from a warehouse (VRAM). Each truck carries 100 units per trip.

- If trucks arrive slower than the shipyard can work → the shipyard sits *idle*, waiting. **Memory-bound.**
- If trucks overwhelm the shipyard → components pile up, shipyard is the limit. **Compute-bound.**

LLM inference with a KV cache is firmly the first scenario. We have a very fast shipyard that spends most of its time waiting for trucks.

---

### 🤔 Think About This

> Suppose you double the GPU's compute speed (better chip) but keep the memory bandwidth the same. Would LLM token generation get twice as fast? Why or why not?

Sit with that question. We will return to it with a precise answer once we have placed inference on the roofline.

## 3. The Mathematics

Now let us make everything above exact.

### 3.1 Naive Inference Cost

For a transformer with $L$ tokens in context and hidden dimension $d$, the attention computation for a single layer performs:

$$\text{Cost}_{\text{naive}}(L) = \underbrace{4 L d}_{\text{project Q,K,V,O}} + \underbrace{2 L^2 d_h \cdot H}_{\text{attention scores + weighted sum}}$$

where $H$ is the number of heads and $d_h = d/H$.

This equation says: take the $L \times d$ token matrix, project it four times (queries, keys, values, output), then compute $L^2$ dot products across all heads. **Computationally, this means the cost grows quadratically in sequence length** — doubling $L$ roughly quadruples the work inside the attention block.

Over $T$ autoregressive steps (generating $T$ tokens), and accounting for the growing context at each step:

$$\text{Total}_{\text{naive}} \approx \sum_{t=0}^{T-1} \text{Cost}_{\text{naive}}(L + t) \approx O(L^2 T + L T^2 + T^3)$$

For large $L$, the dominant term is $O(L^2 T)$. This is catastrophic.

### 3.2 KV Cache — Arithmetic Savings

With a KV cache, at step $t$ we only need to compute the query for the *new* token, look it up against the *cached* keys, and weight the *cached* values:

$$\text{Cost}_{\text{cached}}(t) = \underbrace{2d + d}_{\text{project Q, O}} + \underbrace{2(L+t) d_h \cdot H}_{\text{attend over cache}}$$

The $L^2$ term is gone. Each new token costs $O(L \cdot d)$ instead of $O(L^2)$. Total generation cost is now $O(L T d)$ — **linear** in both context length and number of generated tokens.

### 3.3 Arithmetic Intensity and the Roofline

The **arithmetic intensity** $I$ of a computation is defined as:

$$I = \frac{\text{FLOPs performed}}{\text{Bytes moved from memory}}$$

This equation says: take the total arithmetic work done, divide by the total data moved, and you get a single number that characterises whether the computation is hungry for arithmetic capacity or hungry for memory bandwidth. **Computationally, this means a high $I$ means the chip is busy computing; a low $I$ means the chip is mostly waiting for data.**

The roofline model then says that achieved performance $P$ (in FLOPs/s) is bounded by:

$$P \leq \min\left(I \cdot B_{\text{mem}},\ P_{\text{peak}}\right)$$

where $B_{\text{mem}}$ is memory bandwidth (bytes/s) and $P_{\text{peak}}$ is peak compute throughput (FLOPs/s).

For a single attention head reading a cached KV sequence of length $L$:

$$I_{\text{cached}} = \frac{2L \cdot d_h \text{ FLOPs}}{2L \cdot d_h \cdot \text{bytes\_per\_element}} = \frac{1}{\text{bytes\_per\_element}}$$

For FP16/BF16 (2 bytes), $I_{\text{cached}} \approx 0.5$ FLOPs/byte. A100-class hardware has a ridge point around 200–300 FLOPs/byte. We are *one hundred times* to the left of that ridge. Deep in memory-bound territory.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── Reproduce the arithmetic intensity calculation analytically ──
print("=" * 60)
print("Arithmetic Intensity of KV-Cache Attention Decoding")
print("=" * 60)

d_h   = 128          # head dimension (typical for many models)
L_ctx = 2048         # context length
dtype_bytes = 2      # BF16 / FP16

# FLOPs: for each head, Q·K^T is (1 x d_h) dot (L x d_h) → L mults+adds
# then softmax-weighted sum of V is another L mults+adds
flops_per_head = 2 * L_ctx * d_h * 2   # factor-of-2 for mult+add

# Bytes moved: read K cache (L × d_h) and V cache (L × d_h)
bytes_per_head = 2 * L_ctx * d_h * dtype_bytes

AI = flops_per_head / bytes_per_head

print(f"\nHead dimension d_h         : {d_h}")
print(f"Context length L           : {L_ctx}")
print(f"FLOPs per head (decode)    : {flops_per_head:,}")
print(f"Bytes moved per head       : {bytes_per_head:,}")
print(f"\nArithmetic Intensity       : {AI:.2f} FLOPs/byte")
print(f"\nA100 ridge point           : ~300 FLOPs/byte")
print(f"We are {300/AI:.0f}x to the LEFT of the ridge point.")
print("\n→ Inference is firmly memory-bound.")

## 4. Let's Build It — Component by Component

### 4.1 Simulating Naive vs. Cached FLOPs

Let us build a function that computes the total floating-point operations for both approaches across a range of context lengths and generation counts. This will give us the first half of the picture.

In [ ]:
def naive_total_flops(L: int, T: int, d: int, H: int) -> float:
    """
    Total FLOPs for naive autoregressive inference.

    At each generation step t, we process (L + t) tokens through full attention.
    FLOPs per step:  4*(L+t)*d  (projections)  +  2*(L+t)^2 * d/H * H  (attention)
    Summed over T steps.

    Returns total FLOPs as a float.
    """
    total = 0.0
    for t in range(T):
        ctx = L + t
        proj_flops  = 4 * ctx * d * 2          # Q,K,V,O projections (factor 2: mul+add)
        attn_flops  = 2 * (ctx ** 2) * d * 2   # QK^T + softmax-weighted V
        total += proj_flops + attn_flops
    return total


def cached_total_flops(L: int, T: int, d: int, H: int) -> float:
    """
    Total FLOPs for KV-cache autoregressive inference.

    At each generation step t, only the NEW token's projections are computed,
    but attention still reads the full cache of length (L + t).

    Returns total FLOPs as a float.
    """
    total = 0.0
    for t in range(T):
        ctx = L + t
        proj_flops  = 2 * d * 2                # only new token Q + O projection
        attn_flops  = 2 * ctx * (d // H) * H * 2  # attend over full cache
        total += proj_flops + attn_flops
    return total


# Quick sanity check
L, T, d, H = 512, 100, 1024, 16
print(f"Naive total FLOPs  : {naive_total_flops(L, T, d, H) / 1e9:.2f} GFLOPs")
print(f"Cached total FLOPs : {cached_total_flops(L, T, d, H) / 1e9:.2f} GFLOPs")
speedup = naive_total_flops(L, T, d, H) / cached_total_flops(L, T, d, H)
print(f"Arithmetic speedup : {speedup:.1f}x")

Now let us visualise how this ratio changes as we scale the context length.

In [ ]:
# 📊 Visualization Checkpoint 1 — FLOPs comparison across context lengths

context_lengths = np.arange(128, 4097, 128)
T_gen, d, H = 200, 1024, 16

naive_flops  = [naive_total_flops(L, T_gen, d, H) / 1e12 for L in context_lengths]
cached_flops = [cached_total_flops(L, T_gen, d, H) / 1e12 for L in context_lengths]
speedups     = [n / c for n, c in zip(naive_flops, cached_flops)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(context_lengths, naive_flops,  "tomato",      lw=2.5, label="Naive (no cache)")
ax1.plot(context_lengths, cached_flops, "steelblue",   lw=2.5, label="KV Cache")
ax1.fill_between(context_lengths, cached_flops, naive_flops, alpha=0.15, color="tomato")
ax1.set_xlabel("Context Length (tokens)", fontsize=12)
ax1.set_ylabel("Total FLOPs  (TFLOPs)", fontsize=12)
ax1.set_title("Total Arithmetic Work\n(generating 200 tokens, 1 transformer layer)", fontsize=12)
ax1.legend(fontsize=11); ax1.grid(alpha=0.3)

ax2.plot(context_lengths, speedups, color="darkorchid", lw=2.5)
ax2.axhline(y=1, color="gray", ls="--", lw=1)
ax2.set_xlabel("Context Length (tokens)", fontsize=12)
ax2.set_ylabel("Arithmetic Speedup  (×)", fontsize=12)
ax2.set_title("KV Cache Arithmetic Speedup\nvs. Naive Inference", fontsize=12)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("flops_comparison.png", dpi=120)
plt.show()
print("The shaded region is the work the KV cache *saves* us. Notice how it balloons with context length.")

### 4.2 Memory Footprint of the KV Cache

Now let us model the other side of the coin: how much VRAM the KV cache consumes.

In [ ]:
def kv_cache_bytes(L: int, d: int, H: int, n_layers: int,
                   dtype_bytes: int = 2) -> int:
    """
    Memory footprint of the KV cache in bytes.

    For each layer, we store:
      - Key matrix  : (L, d) elements
      - Value matrix: (L, d) elements
    Across all heads, across all layers.

    dtype_bytes: 2 for FP16/BF16, 4 for FP32.
    """
    per_layer = 2 * L * d * dtype_bytes    # K + V, each (L × d)
    return per_layer * n_layers


# Let us explore across realistic model configurations
configs = {
    "GPT-2 (117M)":     dict(d=768,   H=12, n_layers=12),
    "GPT-2 XL (1.5B)":  dict(d=1600,  H=25, n_layers=48),
    "LLaMA-7B (approx)":dict(d=4096,  H=32, n_layers=32),
    "LLaMA-70B (approx)":dict(d=8192, H=64, n_layers=80),
}

context_lengths_mem = [512, 1024, 2048, 4096, 8192]

print(f"{'Model':<25} {'Context':>8} {'KV Cache (GB)':>14}")
print("-" * 52)
for model_name, cfg in configs.items():
    for L in [2048, 8192]:
        mem_bytes = kv_cache_bytes(L, cfg["d"], cfg["H"], cfg["n_layers"])
        mem_gb    = mem_bytes / 1e9
        print(f"{model_name:<25} {L:>8,}  {mem_gb:>12.2f} GB")
    print()

In [ ]:
# 📊 Visualization Checkpoint 2 — KV cache memory growth

fig, ax = plt.subplots(figsize=(10, 5))

ctx_range = np.arange(256, 8193, 256)
colors_mem = ["#4e9af1", "#f97316", "#10b981", "#a855f7"]

for (model_name, cfg), color in zip(configs.items(), colors_mem):
    mem_gb = [kv_cache_bytes(L, cfg["d"], cfg["H"], cfg["n_layers"]) / 1e9
              for L in ctx_range]
    ax.plot(ctx_range, mem_gb, lw=2.5, color=color, label=model_name)

# Reference lines — typical GPU VRAM sizes
for vram, label, ls in [(24, "24 GB (RTX 3090)", "--"),
                        (40, "40 GB (A100-40)",  "-."),
                        (80, "80 GB (A100-80)",  ":")]:
    ax.axhline(y=vram, color="gray", ls=ls, lw=1.2, label=label)

ax.set_xlabel("Context Length (tokens)", fontsize=12)
ax.set_ylabel("KV Cache Memory (GB)", fontsize=12)
ax.set_title("KV Cache Memory Footprint\nAcross Model Sizes and Context Lengths", fontsize=12)
ax.legend(fontsize=9, loc="upper left"); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("kv_cache_memory.png", dpi=120)
plt.show()
print("Notice how quickly the cache consumes available VRAM for large models and long contexts.")

### 4.3 Bandwidth Demand at Decode Time

Every generation step, the entire KV cache must be *read* from VRAM. Let us model how much bandwidth that demands and compare it to what modern hardware actually provides.

In [ ]:
def decode_bandwidth_demand_GB_per_s(
    L: int, d: int, n_layers: int,
    dtype_bytes: int = 2,
    time_per_token_ms: float = 50.0
) -> float:
    """
    Effective memory bandwidth demanded during a single decode step.

    At every token generation step we must read the entire KV cache.
    bandwidth = cache_size_bytes / time_per_token_seconds
    """
    cache_bytes = kv_cache_bytes(L, d, n_layers=n_layers,
                                 H=1,              # H absorbed into d already
                                 dtype_bytes=dtype_bytes)
    # H=1 trick: kv_cache_bytes counts d as total, not per-head
    # Let us compute directly:
    cache_bytes = 2 * L * d * n_layers * dtype_bytes
    time_s = time_per_token_ms / 1000.0
    return (cache_bytes / 1e9) / time_s   # GB/s


# Typical A100 bandwidth: ~2,000 GB/s
A100_bandwidth = 2_000   # GB/s

print("Bandwidth demanded per decode step (LLaMA-7B approx):")
print(f"{'Context':>10} | {'Cache GB':>10} | {'Demand (GB/s)':>14} | {'% of A100 BW':>14}")
print("-" * 56)

for L in [512, 1024, 2048, 4096, 8192]:
    demand = decode_bandwidth_demand_GB_per_s(L, d=4096, n_layers=32)
    cache_gb = 2 * L * 4096 * 32 * 2 / 1e9
    pct = 100 * demand / A100_bandwidth
    flag = " ⚠️" if pct > 80 else ""
    print(f"{L:>10,} | {cache_gb:>10.2f} | {demand:>14.1f} | {pct:>13.1f}%{flag}")

print(f"\nA100 peak bandwidth: {A100_bandwidth:,} GB/s")

## 5. 🔧 Your Turn

Time to test your understanding. We want you to implement two functions that combine everything we have discussed. Take your time — the hints walk you through it step by step.

In [ ]:
def arithmetic_intensity_decode(
    L: int,
    d: int,
    H: int,
    n_layers: int,
    dtype_bytes: int = 2
) -> float:
    """
    Compute the arithmetic intensity (FLOPs / byte) of a single
    autoregressive decode step WITH a KV cache.

    Remember:
      - FLOPs: for each layer and each head, the new query token
               attends over L cached keys → 2 * L * d_h FLOPs for QK^T
               and another 2 * L * d_h FLOPs for the weighted V sum.
               Also include Q and O projection FLOPs: 2 * d * d for each.
      - Bytes: we must READ the KV cache (K and V matrices, all layers),
               and READ the model weight matrices for Q, K, V, O projections.
               For simplicity, focus on the KV cache read dominance:
               bytes = 2 * L * (d_h * H) * n_layers * dtype_bytes  (K)
                     + 2 * L * (d_h * H) * n_layers * dtype_bytes  (V)
               (K and V, across all layers)

    Returns:
        float: arithmetic intensity in FLOPs per byte
    """
    # ============ TODO ============
    # Step 1: Compute d_h = d // H

    # Step 2: Compute total attention FLOPs across all heads and all layers
    #         (Q·K^T + softmax-weighted V, for the single new query token)
    #         Hint: per head per layer = 2 * 2 * L * d_h  (two dot products)

    # Step 3: Add projection FLOPs (Q and O projections, per layer)
    #         Hint: each projection is a (d,) vector × (d, d) matrix = 2 * d * d

    # Step 4: Compute bytes moved — just the KV cache reads
    #         Hint: K cache = n_layers * L * d * dtype_bytes
    #               V cache = n_layers * L * d * dtype_bytes

    # Step 5: Return total_flops / total_bytes
    # ============ END TODO ========
    result = None  # YOUR CODE HERE
    return result


def roofline_achieved_gflops(
    arithmetic_intensity: float,
    bandwidth_GBs: float,
    peak_gflops: float
) -> float:
    """
    Given the arithmetic intensity of a workload and the hardware specs,
    return the achieved performance predicted by the roofline model.

    The roofline says:
        achieved = min(arithmetic_intensity * bandwidth, peak_compute)

    Args:
        arithmetic_intensity : FLOPs per byte
        bandwidth_GBs        : hardware memory bandwidth in GB/s
        peak_gflops          : hardware peak compute in GFLOPs/s

    Returns:
        float: achieved GFLOPs/s
    """
    # ============ TODO ============
    # Step 1: Compute memory-bound ceiling = arithmetic_intensity * bandwidth_GBs
    #         (bandwidth is in GB/s = 10^9 bytes/s; intensity is FLOPs/byte
    #          → result is in GFLOPs/s directly)

    # Step 2: Return min(memory_bound_ceiling, peak_gflops)
    # ============ END TODO ========
    result = None  # YOUR CODE HERE
    return result

Now let us verify your implementations with a set of assertions:

In [ ]:
# ✅ Verification Cell

# Test arithmetic_intensity_decode
# For L=1024, d=512, H=8, n_layers=1, dtype=2:
#   d_h = 64
#   attn FLOPs per head per layer = 4 * 1024 * 64 = 262144
#   total attn FLOPs (8 heads, 1 layer) = 2,097,152
#   proj FLOPs (Q+O, 1 layer) = 2 * 2 * 512 * 512 = 1,048,576
#   total FLOPs ≈ 3,145,728
#   KV bytes = 2 * 1024 * 512 * 1 * 2 = 2,097,152
#   AI ≈ 3145728 / 2097152 ≈ 1.5

ai_test = arithmetic_intensity_decode(L=1024, d=512, H=8, n_layers=1, dtype_bytes=2)
assert ai_test is not None, "Function returned None — did you forget to implement it?"
assert 1.0 < ai_test < 2.5, (
    f"Expected AI between 1.0 and 2.5, got {ai_test:.3f}. "
    "Check your FLOPs and bytes calculations."
)
print(f"✅ arithmetic_intensity_decode returned {ai_test:.3f} FLOPs/byte  (expected ~1.5)")

# Test roofline_achieved_gflops
r1 = roofline_achieved_gflops(0.5, 2000, 312_000)
assert r1 is not None, "Function returned None — implement it!"
assert abs(r1 - 1000.0) < 1.0, f"Expected 1000.0 GFLOPs/s, got {r1}"
print(f"✅ roofline (memory-bound case): {r1:.1f} GFLOPs/s  (expected 1000.0)")

r2 = roofline_achieved_gflops(1000, 2000, 312_000)
assert abs(r2 - 312_000) < 1.0, f"Expected 312000 GFLOPs/s, got {r2}"
print(f"✅ roofline (compute-bound case): {r2:.1f} GFLOPs/s  (expected 312000.0)")

print("\n🎉 All checks passed! Let us build the full roofline diagram.")

## 6. Putting It All Together

We now have all the ingredients. Let us assemble the complete picture: the roofline diagram annotated with real workloads, and a unified view of how the KV cache trades arithmetic work for memory pressure.

In [ ]:
# Reference hardware specs (A100 SXM 80GB BF16)
HW = dict(
    bandwidth_GBs = 2_000,    # GB/s
    peak_gflops   = 312_000,  # GFLOPs/s (BF16 Tensor Core)
    ridge_point   = 312_000 / 2_000   # FLOPs/byte where we transition
)
print(f"A100 ridge point: {HW['ridge_point']:.0f} FLOPs/byte")
print("Any workload with AI < this is memory-bound.")

In [ ]:
# Compute AI for several realistic workloads

workloads = {}

# LLaMA-7B style decode at different context lengths
for L in [512, 1024, 2048, 4096]:
    label = f"LLaMA-7B decode  L={L}"
    ai = arithmetic_intensity_decode(L=L, d=4096, H=32, n_layers=32, dtype_bytes=2)
    ach = roofline_achieved_gflops(ai, HW["bandwidth_GBs"], HW["peak_gflops"])
    workloads[label] = dict(ai=ai, achieved=ach, color="crimson", marker="o")

# Large-batch prefill (many tokens processed together — more compute-bound)
for batch in [8, 64]:
    # Prefill processes L tokens at once → intensity scales with batch/sequence
    # Rough estimate: batch processing moves us right on the roofline
    ai_prefill = arithmetic_intensity_decode(L=2048, d=4096, H=32,
                                              n_layers=32, dtype_bytes=2) * batch
    ai_prefill = min(ai_prefill, HW["ridge_point"] * 1.5)  # cap for realism
    ach = roofline_achieved_gflops(ai_prefill, HW["bandwidth_GBs"], HW["peak_gflops"])
    label = f"Prefill  batch={batch}, L=2048"
    workloads[label] = dict(ai=ai_prefill, achieved=ach, color="steelblue", marker="s")

for k, v in workloads.items():
    region = "MEMORY-BOUND" if v["ai"] < HW["ridge_point"] else "COMPUTE-BOUND"
    print(f"{k:<40}  AI={v['ai']:7.2f}  Achieved={v['achieved']:10.1f} GFLOPs/s  [{region}]")

In [ ]:
# 📊 Visualization Checkpoint 3 — The Full Roofline Diagram

fig, ax = plt.subplots(figsize=(11, 6))
ax.set_xscale("log"); ax.set_yscale("log")

# Draw roofline
x_range = np.logspace(-1, 4, 1000)
roof    = np.minimum(HW["bandwidth_GBs"] * x_range, HW["peak_gflops"])
ax.plot(x_range, roof, "k-", lw=3, label="A100 Roofline", zorder=3)

# Shade memory-bound and compute-bound regions
ridge = HW["ridge_point"]
x_mem = x_range[x_range <= ridge]
x_cmp = x_range[x_range >= ridge]
ax.fill_between(x_mem, 1, HW["bandwidth_GBs"] * x_mem,
                alpha=0.10, color="tomato",    label="Memory-Bound Region")
ax.fill_between(x_cmp, 1, HW["peak_gflops"],
                alpha=0.10, color="steelblue", label="Compute-Bound Region")

# Labels for regions
ax.text(0.3,  3000,   "Memory-Bound\n(bandwidth limited)",
        fontsize=11, color="tomato",    ha="center")
ax.text(1500, 180000, "Compute-Bound\n(arithmetic limited)",
        fontsize=11, color="steelblue", ha="center")

# Ridge point
ax.axvline(x=ridge, color="gray", ls="--", lw=1.5, alpha=0.6)
ax.text(ridge * 1.08, 500, f"Ridge point\n{ridge:.0f} FLOPs/byte",
        fontsize=9, color="gray")

# Annotate hardware ceilings
ax.axhline(y=HW["peak_gflops"], color="navy", ls=":", lw=1.2, alpha=0.5)
ax.text(1.2e3, HW["peak_gflops"] * 1.05, f"Peak Compute: {HW['peak_gflops']//1000}K GFLOPs/s",
        fontsize=9, color="navy")

# Plot workloads
decode_plotted = False
prefill_plotted = False
for label, v in workloads.items():
    is_decode = "decode" in label
    marker_label = ("LLaMA-7B KV-Cache Decode (various L)" if is_decode and not decode_plotted
                    else ("Prefill (larger batch)" if not is_decode and not prefill_plotted
                          else "_nolegend_"))
    if is_decode: decode_plotted = True
    else:         prefill_plotted = True

    ax.scatter(v["ai"], v["achieved"],
               s=160, color=v["color"], marker=v["marker"],
               zorder=5, label=marker_label, edgecolors="black", linewidths=0.7)

    # Small annotation offset
    yoff = 1.35 if is_decode else 0.6
    ax.annotate(label.split("  ")[-1],
                xy=(v["ai"], v["achieved"]),
                xytext=(v["ai"] * 1.15, v["achieved"] * yoff),
                fontsize=8, color=v["color"],
                arrowprops=dict(arrowstyle="-", color=v["color"], lw=0.8))

ax.set_xlabel("Arithmetic Intensity  (FLOPs / byte)", fontsize=13)
ax.set_ylabel("Achieved Performance  (GFLOPs/s)", fontsize=13)
ax.set_title("Roofline Model — A100 SXM (BF16)\nAnnotated with LLM Inference Workloads",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10, loc="upper left")
ax.grid(True, which="both", alpha=0.2)
ax.set_xlim(0.1, 5000); ax.set_ylim(100, 600_000)

plt.tight_layout()
plt.savefig("full_roofline.png", dpi=140)
plt.show()

print("\nKey observation: ALL decode points cluster far left — deeply memory-bound.")
print("Prefill with larger batches migrates toward the ridge point.")

## 7. The Complete Inference Timeline

Let us now simulate the full token-generation timeline: prefill phase followed by decode phase, showing where time goes.

In [ ]:
def simulate_inference_timeline(
    L_prompt: int,
    T_generate: int,
    d: int,
    H: int,
    n_layers: int,
    bandwidth_GBs: float = 2_000,
    peak_gflops: float   = 312_000,
    dtype_bytes: int     = 2
):
    """
    Simulate token-by-token inference and record per-step time and bottleneck.
    Returns arrays of times (ms) and arithmetic intensities per step.
    """
    times_ms = []
    intensities = []
    phases = []

    # ── Prefill phase (process all L_prompt tokens in parallel) ──
    # Prefill AI is roughly: 2 * L^2 * d FLOPs / (L * d * bytes)
    #                       = 2 * L / bytes_per_element
    prefill_flops = 2 * (L_prompt ** 2) * d * n_layers * 2   # rough
    prefill_bytes = 4 * L_prompt * d * n_layers * dtype_bytes  # Q,K,V,O reads
    prefill_AI    = prefill_flops / (prefill_bytes * 1e9)      # FLOPs/byte
    prefill_achieved = min(prefill_AI * bandwidth_GBs, peak_gflops)  # GFLOPs/s
    prefill_time_ms  = (prefill_flops / 1e9) / prefill_achieved * 1000

    times_ms.append(prefill_time_ms)
    intensities.append(prefill_AI)
    phases.append("prefill")

    # ── Decode phase (one token at a time) ──
    for t in range(T_generate):
        L_cur = L_prompt + t
        ai    = arithmetic_intensity_decode(L_cur, d, H, n_layers, dtype_bytes)

        # Decode FLOPs per step (attention + projections, 1 new token)
        d_h       = d // H
        attn_flops = 4 * L_cur * d_h * H * n_layers * 2
        proj_flops = 2 * d * d * n_layers * 2   # Q,K,V,O projections
        total_flops = (attn_flops + proj_flops) / 1e9  # GFLOPs

        achieved  = roofline_achieved_gflops(ai, bandwidth_GBs, peak_gflops)
        step_time = total_flops / achieved * 1000   # ms

        times_ms.append(step_time)
        intensities.append(ai)
        phases.append("decode")

    return np.array(times_ms), np.array(intensities), phases


# Run simulation for a GPT-2-Large-like model
times, intensities, phases = simulate_inference_timeline(
    L_prompt=1024, T_generate=200, d=1280, H=20, n_layers=36
)

print(f"Prefill time          : {times[0]:.1f} ms")
print(f"Mean decode time/tok  : {times[1:].mean():.2f} ms")
print(f"Total decode time     : {times[1:].sum():.0f} ms  ({times[1:].sum()/1000:.1f} s)")
print(f"Mean decode AI        : {intensities[1:].mean():.2f} FLOPs/byte")
print(f"Ridge point           : {312_000 / 2_000:.0f} FLOPs/byte")
print(f"We are {(312_000/2_000) / intensities[1:].mean():.0f}x below the ridge point.")

In [ ]:
# 📊 Visualization Checkpoint 4 — Inference Timeline

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=False)

# ── Top panel: per-step time ──
ax = axes[0]
steps = np.arange(len(times))
decode_mask  = np.array([p == "decode"  for p in phases])
prefill_mask = np.array([p == "prefill" for p in phases])

ax.bar(steps[prefill_mask], times[prefill_mask],
       color="steelblue", label="Prefill", width=1.5)
ax.bar(steps[decode_mask], times[decode_mask],
       color="tomato", alpha=0.8, label="Decode (per token)", width=0.9)

ax.axhline(y=times[1:].mean(), color="darkred", ls="--", lw=1.5,
           label=f"Mean decode time ({times[1:].mean():.2f} ms)")
ax.set_ylabel("Time per step (ms)", fontsize=12)
ax.set_title("Inference Timeline: Prefill + Autoregressive Decode", fontsize=13)
ax.legend(fontsize=10); ax.grid(axis="y", alpha=0.3)
ax.set_xlim(-2, len(times) + 1)

# ── Bottom panel: arithmetic intensity per step ──
ax2 = axes[1]
ax2.plot(steps[decode_mask], intensities[decode_mask],
         color="darkorchid", lw=2, label="Decode AI")
ax2.scatter(steps[prefill_mask], intensities[prefill_mask],
            s=150, color="steelblue", zorder=5, label="Prefill AI")
ax2.axhline(y=312_000/2_000, color="gray", ls="--", lw=1.5,
            label=f"Ridge point ({312_000//2_000} FLOPs/byte)")
ax2.set_ylabel("Arithmetic Intensity\n(FLOPs / byte)", fontsize=12)
ax2.set_xlabel("Generation Step", fontsize=12)
ax2.set_title("Arithmetic Intensity Across Inference Steps", fontsize=12)
ax2.legend(fontsize=10); ax2.grid(alpha=0.3)
ax2.set_yscale("log")

plt.tight_layout()
plt.savefig("inference_timeline.png", dpi=130)
plt.show()
print("The decode AI is essentially flat and stays orders of magnitude below the ridge.")

## 8. 🎯 Final Output — The Full Picture Dashboard

Let us now produce our capstone visualisation: a four-panel dashboard that tells the complete story we have built together, from arithmetic savings to memory pressure to hardware implications.

In [ ]:
# ─────────────────────────────────────────────────────────────
#  THE FULL PICTURE — Four-panel summary dashboard
# ─────────────────────────────────────────────────────────────
import matplotlib.gridspec as gridspec
import matplotlib.ticker as ticker

fig = plt.figure(figsize=(16, 11), facecolor="#0f0f1a")
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.40, wspace=0.35)

panel_bg   = "#1a1a2e"
text_color = "#e8e8f0"
grid_color = "#2a2a3e"

# ── Shared context ──
L_vals = np.arange(256, 4097, 128)
T_gen  = 200
d, H, n_layers = 4096, 32, 32   # LLaMA-7B proxy

# ─────────────────────────────────────────
#  Panel 1 (top-left): FLOPs saved by KV cache
# ─────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.set_facecolor(panel_bg)

n_fl = [naive_total_flops(L, T_gen, d, H) / 1e12 for L in L_vals]
c_fl = [cached_total_flops(L, T_gen, d, H) / 1e12 for L in L_vals]

ax1.fill_between(L_vals, c_fl, n_fl, alpha=0.30, color="#ff6b6b")
ax1.plot(L_vals, n_fl,  color="#ff6b6b",  lw=2.5, label="Naive (no cache)")
ax1.plot(L_vals, c_fl,  color="#4ecdc4",  lw=2.5, label="KV Cache")
ax1.set_xlabel("Context Length (tokens)", color=text_color, fontsize=10)
ax1.set_ylabel("Total FLOPs  (TFLOPs)",   color=text_color, fontsize=10)
ax1.set_title("① Arithmetic Savings\nfrom KV Cache", color=text_color, fontsize=11,
              fontweight="bold")
ax1.tick_params(colors=text_color); ax1.grid(color=grid_color, alpha=0.5)
ax1.spines[:].set_color(grid_color)
ax1.legend(fontsize=9, facecolor=panel_bg, labelcolor=text_color)

# ─────────────────────────────────────────
#  Panel 2 (top-right): KV cache memory footprint
# ─────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_facecolor(panel_bg)

palette = ["#ffd93d", "#6bcb77", "#4d96ff", "#ff6b6b"]
small_cfgs = [
    ("7B  (d=4096, L=32)",  dict(d=4096,  H=32, n_layers=32)),
    ("13B (d=5120, L=40)",  dict(d=5120,  H=40, n_layers=40)),
    ("34B (d=7168, L=48)",  dict(d=7168,  H=56, n_layers=48)),
    ("70B (d=8192, L=80)",  dict(d=8192,  H=64, n_layers=80)),
]
for (name, cfg), col in zip(small_cfgs, palette):
    mem = [kv_cache_bytes(L, cfg["d"], cfg["H"], cfg["n_layers"]) / 1e9
           for L in L_vals]
    ax2.plot(L_vals, mem, color=col, lw=2.5, label=name)

for vram, ls_ in [(24, "--"), (40, "-."), (80, ":")]:
    ax2.axhline(y=vram, color="white", ls=ls_, lw=1, alpha=0.35)
    ax2.text(L_vals[-1] * 0.72, vram + 0.5, f"{vram}GB VRAM",
             color="white", alpha=0.55, fontsize=8)

ax2.set_xlabel("Context Length (tokens)", color=text_color, fontsize=10)
ax2.set_ylabel("KV Cache Memory (GB)",    color=text_color, fontsize=10)
ax2.set_title("② Memory Footprint\nof KV Cache", color=text_color, fontsize=11,
              fontweight="bold")
ax2.tick_params(colors=text_color); ax2.grid(color=grid_color, alpha=0.5)
ax2.spines[:].set_color(grid_color)
ax2.legend(fontsize=8, facecolor=panel_bg, labelcolor=text_color)

# ─────────────────────────────────────────
#  Panel 3 (bottom-left): Roofline with workloads
# ─────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
ax3.set_facecolor(panel_bg)
ax3.set_xscale("log"); ax3.set_yscale("log")

x_r   = np.logspace(-1, 4, 800)
roof_ = np.minimum(2_000 * x_r, 312_000)
ax3.plot(x_r, roof_, color="white", lw=2.5, zorder=3)

x_mem_ = x_r[x_r <= 156]
ax3.fill_between(x_mem_, 1, 2_000 * x_mem_, alpha=0.15, color="#ff6b6b")
x_cmp_ = x_r[x_r >= 156]
ax3.fill_between(x_cmp_, 1, 312_000,         alpha=0.15, color="#4d96ff")

ax3.text(0.15, 80,    "Memory\nBound",  color="#ff6b6b", fontsize=10, fontweight="bold")
ax3.text(500,  180000,"Compute\nBound", color="#4d96ff", fontsize=10, fontweight="bold")

# Decode points at different L
for i, L_pt in enumerate([512, 1024, 2048, 4096]):
    ai_pt = arithmetic_intensity_decode(L_pt, d, H, n_layers, 2)
    ach   = roofline_achieved_gflops(ai_pt, 2_000, 312_000)
    ax3.scatter(ai_pt, ach, s=120, color="#ff6b6b", zorder=6,
                edgecolors="white", linewidths=0.8)
    ax3.text(ai_pt * 1.2, ach * 1.3, f"L={L_pt}", color="#ff6b6b", fontsize=8)

ax3.set_xlabel("Arithmetic Intensity  (FLOPs/byte)", color=text_color, fontsize=10)
ax3.set_ylabel("Performance  (GFLOPs/s)",            color=text_color, fontsize=10)
ax3.set_title("③ Roofline Model\nA100 SXM + KV-Cache Decode", color=text_color,
              fontsize=11, fontweight="bold")
ax3.tick_params(colors=text_color); ax3.grid(color=grid_color, alpha=0.3, which="both")
ax3.spines[:].set_color(grid_color)
ax3.set_xlim(0.1, 5000); ax3.set_ylim(100, 600_000)

# ─────────────────────────────────────────
#  Panel 4 (bottom-right): Cumulative time breakdown
# ─────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
ax4.set_facecolor(panel_bg)

ctx_sizes = [512, 1024, 2048, 4096]
T_g       = 100
prefill_t, decode_t = [], []

for L_s in ctx_sizes:
    t_arr, _, ph_arr = simulate_inference_timeline(
        L_s, T_g, d=1280, H=20, n_layers=36
    )
    prefill_t.append(t_arr[0])
    decode_t.append(t_arr[1:].sum())

x_pos = np.arange(len(ctx_sizes))
bars_p = ax4.bar(x_pos, prefill_t, color="#4d96ff", label="Prefill",   width=0.55)
bars_d = ax4.bar(x_pos, decode_t,  bottom=prefill_t,
                 color="#ff6b6b", label="Decode (100 tok)", width=0.55, alpha=0.9)

ax4.set_xticks(x_pos)
ax4.set_xticklabels([f"L={c}" for c in ctx_sizes])
ax4.set_ylabel("Time (ms)", color=text_color, fontsize=10)
ax4.set_title("④ Inference Time Breakdown\nPrefill vs. Decode Phase", color=text_color,
              fontsize=11, fontweight="bold")
ax4.tick_params(colors=text_color); ax4.grid(axis="y", color=grid_color, alpha=0.5)
ax4.spines[:].set_color(grid_color)
ax4.legend(fontsize=9, facecolor=panel_bg, labelcolor=text_color)

# ─────────────────────────────────────────
#  Main title
# ─────────────────────────────────────────
fig.suptitle("The Full Picture — KV Cache: From Arithmetic Savings to Hardware Bottlenecks",
             color=text_color, fontsize=15, fontweight="bold", y=0.98)

plt.savefig("full_picture_dashboard.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print("\n✅ Dashboard saved to full_picture_dashboard.png")
print("This is the full picture. Every panel tells part of the same story.")

## 9. Reflection and Next Steps

Take a moment to look back at what we have built — and what it means.

### 🤔 Reflection Questions

Let us revisit the question we posed at the very beginning of this notebook:

**1. If you double the GPU's compute speed but keep bandwidth fixed, does decode speed double?**

No. Because decode is memory-bound, *compute speed is not the bottleneck*. Doubling compute capacity while leaving bandwidth unchanged leaves the decode time almost unchanged. The shipyard can work faster, but the trucks still arrive at the same rate.

**2. What does the KV cache actually trade?**

It trades *arithmetic work* (recomputing K and V on every step) for *memory work* (reading the cached K and V from VRAM on every step). For long contexts, the memory work grows with context length, while the arithmetic work per step stays relatively small. Both grow, but we shift which resource is stressed.

**3. Why does prefill feel fast and decode feel slow?**

Prefill processes many tokens in parallel — it is compute-bound (or at least closer to the ridge point). Decode generates one token at a time — there is very little arithmetic per step, but the memory system must deliver the entire cache every single time. That is what makes decode feel sluggish.

**4. What is the arrow that points toward solutions?**

The key insight from the roofline is: to speed up memory-bound inference, either *reduce the bytes that must be moved* (techniques like GQA, MQA, quantisation, cache compression) or *increase the arithmetic done per byte moved* (batching multiple requests together, speculative decoding). Both strategies move the operating point rightward on the roofline.

---

### 🏆 Optional Challenges

If you want to go further, here are three directions that will sharpen your understanding considerably:

**Challenge 1 — Group-Query Attention (GQA)**
Modify `arithmetic_intensity_decode` to model GQA, where $G$ query heads share one key-value head. How does this change the bytes moved? Re-plot the roofline with GQA points alongside MHA points.

**Challenge 2 — Quantised KV Cache**
In our model, `dtype_bytes=2` (BF16). What happens if you set `dtype_bytes=1` (INT8 quantisation of the cache)? Compute the new AI values and see how far right they move on the roofline. What is the practical tradeoff?

**Challenge 3 — Batch Decoding**
Our model generates for a single sequence. In practice, servers batch $B$ requests together. Extend `arithmetic_intensity_decode` to accept a batch size $B$. Show that larger batches move the operating point toward the ridge, and find the minimum $B$ needed to become compute-bound on an A100.

---

### 🗺️ The Arc We Have Traced

```
Naive O(L²) inference
        ↓  KV Cache eliminates redundant computation
O(L) per step — arithmetic bottleneck solved
        ↓  but memory footprint grows with L
VRAM fills with cached K and V tensors
        ↓  every decode step reads the entire cache
Memory bandwidth becomes the new bottleneck
        ↓  roofline model shows we are deep in memory-bound territory
Engineering solutions: GQA, MQA, quantisation, batching, speculative decoding
```

That is the full picture. The KV cache is genuinely brilliant — it is one of the most impactful engineering decisions in modern LLM deployment. But understanding *both* what it saves and what it costs is what separates an informed engineer from a casual observer.

In [ ]:
# 🎉 Final summary printout
print("=" * 65)
print("  THE FULL PICTURE — Summary of Key Numbers")
print("=" * 65)

L_summary  = 2048
d_summary  = 4096
H_summary  = 32
nl_summary = 32

ai_val  = arithmetic_intensity_decode(L_summary, d_summary, H_summary, nl_summary, 2)
ach_val = roofline_achieved_gflops(ai_val, 2_000, 312_000)
util    = 100 * ach_val / 312_000
mem_gb  = kv_cache_bytes(L_summary, d_summary, H_summary, nl_summary, 2) / 1e9
speedup_val = naive_total_flops(L_summary, 200, d_summary, H_summary) / \
              cached_total_flops(L_summary, 200, d_summary, H_summary)

print(f"\n  Model proxy     : LLaMA-7B  (d={d_summary}, H={H_summary}, layers={nl_summary})")
print(f"  Context length  : {L_summary:,} tokens")
print(f"\n  Arithmetic intensity (decode)  : {ai_val:.2f}  FLOPs/byte")
print(f"  Ridge point (A100)             : {312_000 // 2_000:,}  FLOPs/byte")
print(f"  We are {(312_000//2_000)/ai_val:.0f}x below the ridge point")
print(f"\n  Achieved performance           : {ach_val:,.0f}  GFLOPs/s")
print(f"  Hardware utilisation           : {util:.2f}%  of peak compute")
print(f"\n  KV cache size (BF16)           : {mem_gb:.2f}  GB")
print(f"  Arithmetic speedup vs. naive   : {speedup_val:.1f}×")
print(f"\n  Bottleneck                     : MEMORY BANDWIDTH")
print(f"  Fix direction                  : ↓ bytes moved  OR  ↑ batch size")
print("\n" + "=" * 65)
print("  You now understand not just WHAT the KV cache does,")
print("  but WHERE the bottleneck moves — and why.")
print("=" * 65)